# Momants sentimentclassificatie

Deze notebook is een leeromgeving en prototype. We laden berichten uit een lokaal CSV-bestand, selecteren alleen bezoekersberichten, bepalen hun sentiment en maken één samenvatting per gesprek.

**Scope:** alleen sentiment. Geen onderwerpclassificatie, urgentiemodel, live endpoint of database-opslag.

> **Privacyregel:** gebruik hier uitsluitend nep- of testgesprekken. Echte bezoekers kunnen telefoonnummers in `text` typen; die worden in deze versie nog niet verwijderd.

## Stap 1 — Bibliotheken laden

`pandas` verwerkt de tabel. `pipeline` laadt het Hugging Face-model. `Path` helpt om bestanden op een duidelijke manier te vinden.

In [1]:
from pathlib import Path
import re

import pandas as pd
from transformers import pipeline

/home/runner/workspace/.pythonlibs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stap 2 — Instellingen op één plek

Het model-ID en de vertaling van modellabels staan los van de rest van de code. Daardoor kunnen we later een ander model gebruiken door alleen deze instellingen en eventueel de mapping te wijzigen.

In [2]:
MODEL_ID = "tabularisai/multilingual-sentiment-analysis"

LABEL_MAPPING = {
    "Very Positive": "Positief",
    "Positive": "Positief",
    "Neutral": "Neutraal (taakgericht)",
    "Negative": "Negatief (gefrustreerd)",
    "Very Negative": "Boos (paniek)",
}

TOEGESTANE_KOLOMMEN = [
    "created_at",
    "text",
    "from_agent",
    "message_type",
    "conversation_id",
    "agent_id",
]

# Laat dit alleen op True staan voor nep- of testdata.
IK_BEVESTIG_DAT_DIT_TESTDATA_IS = True

## Stap 3 — De databron kiezen

Voor nu verwijst `BRON` naar het meegeleverde synthetische CSV-bestand. Later kan alleen deze regel naar de endpoint-URL wijzen; de rest van de notebook blijft hetzelfde.

In [3]:
huidige_map = Path.cwd()
projectmap = huidige_map.parent if huidige_map.name == "notebooks" else huidige_map
BRON = projectmap / "data" / "voorbeeld_gesprekken.csv"

print(f"Databron: {BRON}")

Databron: /home/runner/workspace/data/voorbeeld_gesprekken.csv


## Stap 4 — Data veilig inladen

De functie hieronder is de enige ingang voor data. Met `usecols` worden uitsluitend de zes toegestane kolommen ingelezen. Privacygevoelige kolommen zoals `raw_json`, `chat_sender`, `media`, `media_url` en `file` komen daardoor nooit in het DataFrame terecht.

In [4]:
def laad_gesprekken(bron):
    """Laad alleen toegestane kolommen uit een lokale CSV-bron."""
    if not IK_BEVESTIG_DAT_DIT_TESTDATA_IS:
        raise ValueError(
            "Stop: deze notebook mag nog uitsluitend nep- of testdata verwerken."
        )

    # Nu gebruiken we een lokaal pad. De rest van de notebook kent de bron niet.
    data = pd.read_csv(
        bron,
        usecols=lambda kolom: kolom in TOEGESTANE_KOLOMMEN,
    )

    ontbrekend = set(TOEGESTANE_KOLOMMEN) - set(data.columns)
    if ontbrekend:
        raise ValueError(f"De databron mist verplichte kolommen: {sorted(ontbrekend)}")

    data = data[TOEGESTANE_KOLOMMEN].copy()
    data["created_at"] = pd.to_datetime(data["created_at"], errors="coerce", utc=True)

    # CSV-bestanden kunnen booleans als tekst bevatten. We maken ze betrouwbaar True/False.
    data["from_agent"] = (
        data["from_agent"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

    if data["created_at"].isna().any():
        raise ValueError("Minstens één created_at-waarde kon niet als datum worden gelezen.")
    if data["from_agent"].isna().any():
        raise ValueError("from_agent mag alleen True of False bevatten.")

    # Zodra het endpoint bestaat: pd.read_csv(url), met een API-key uit
    # Replit Secrets (nooit in de code), een `since`-parameter zodat je
    # alleen nieuwe gesprekken ophaalt, en in blokken (bv. 1000 rijen)
    # zodat één verzoek niet vastloopt bij veel data.

    return data


berichten = laad_gesprekken(BRON)
print(f"{len(berichten)} berichtrijen veilig ingeladen.")
berichten.head()

12 berichtrijen veilig ingeladen.


,created_at,text,from_agent,message_type,conversation_id,agent_id
0,2026-08-01 09:00:00+00:00,Goedemorgen ik heb een vraag over mijn afspraak,False,text,test-gesprek-001,NaN
1,2026-08-01 09:00:15+00:00,Natuurlijk waar kan ik mee helpen?,True,text,test-gesprek-001,test-agent
2,2026-08-01 09:01:00+00:00,Ik zie hem nu staan dank je wel!,False,text,test-gesprek-001,NaN
3,2026-08-01 10:00:00+00:00,Waarom werkt de bevestigingslink niet?,False,text,test-gesprek-002,NaN
4,2026-08-01 10:00:20+00:00,Ik kijk het voor je na.,True,text,test-gesprek-002,test-agent


## Stap 5 — Alleen bruikbare bezoekersberichten kiezen

We sorteren eerst per gesprek en tijdstip. Daarna houden we alleen regels over die van de bezoeker komen en echte tekst bevatten. Lege tekst en een kale URL zeggen niets bruikbaars over sentiment.

In [5]:
def is_kale_url(tekst):
    """Geef True terug als de volledige tekst alleen een webadres is."""
    if pd.isna(tekst):
        return False
    return bool(re.fullmatch(r"(?:https?://|www\.)\S+", str(tekst).strip(), flags=re.I))


gesorteerd = berichten.sort_values(["conversation_id", "created_at"]).copy()
tekst_schoon = gesorteerd["text"].fillna("").astype(str).str.strip()

bezoekersberichten = gesorteerd.loc[
    (~gesorteerd["from_agent"])
    & tekst_schoon.ne("")
    & ~gesorteerd["text"].apply(is_kale_url)
].copy()

bezoekersberichten["text"] = bezoekersberichten["text"].astype(str).str.strip()

print(
    f"{len(bezoekersberichten)} bruikbare bezoekersberichten "
    f"in {bezoekersberichten['conversation_id'].nunique()} gesprekken."
)
bezoekersberichten[["conversation_id", "created_at", "text"]].head(10)

7 bruikbare bezoekersberichten in 5 gesprekken.


,conversation_id,created_at,text
0,test-gesprek-001,2026-08-01 09:00:00+00:00,Goedemorgen ik heb een vraag over mijn afspraak
2,test-gesprek-001,2026-08-01 09:01:00+00:00,Ik zie hem nu staan dank je wel!
3,test-gesprek-002,2026-08-01 10:00:00+00:00,Waarom werkt de bevestigingslink niet?
5,test-gesprek-002,2026-08-01 10:02:00+00:00,Het lukt nog steeds niet en dit is erg frustre...
6,test-gesprek-003,2026-08-01 11:00:00+00:00,HELP mijn gegevens zijn verdwenen en niemand r...
8,test-gesprek-004,2026-08-01 12:00:00+00:00,Wat zijn jullie openingstijden?
11,test-gesprek-005,2026-08-01 13:00:30+00:00,De uitleg was duidelijk en ik kan verder bedankt!


## Stap 6 — Het sentimentmodel laden

De eerste keer downloadt Hugging Face het model. Dat kan enkele minuten duren. Daarna gebruikt de pipeline het lokaal opgeslagen model.

In [6]:
sentiment_model = pipeline(
    task="text-classification",
    model=MODEL_ID,
)

print(f"Model geladen: {MODEL_ID}")

Loading weights:   0%|                                 | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 104/104 [00:00<00:00, 2842.43it/s]

Model geladen: tabularisai/multilingual-sentiment-analysis


## Stap 7 — Ieder bezoekersbericht classificeren

Het model geeft per bericht een label en een score tussen 0 en 1. De losse mapping vertaalt het modellabel naar de vier Momants-waarden.

In [7]:
def classificeer_bericht(tekst):
    """Bepaal het Momants-sentiment en de modelzekerheid van één bericht."""
    modeluitkomst = sentiment_model(tekst, truncation=True)[0]
    modellabel = modeluitkomst["label"].strip()

    if modellabel not in LABEL_MAPPING:
        raise ValueError(
            f"Onbekend modellabel {modellabel!r}. Werk LABEL_MAPPING bovenaan bij."
        )

    return pd.Series(
        {
            "modellabel": modellabel,
            "sentiment_bericht": LABEL_MAPPING[modellabel],
            "zekerheid_bericht": float(modeluitkomst["score"]),
        }
    )


bericht_scores = bezoekersberichten["text"].apply(classificeer_bericht)
bezoekersberichten = pd.concat(
    [bezoekersberichten.reset_index(drop=True), bericht_scores.reset_index(drop=True)],
    axis=1,
)

bezoekersberichten[
    ["conversation_id", "text", "sentiment_bericht", "zekerheid_bericht"]
].head(10)

,conversation_id,text,sentiment_bericht,zekerheid_bericht
0,test-gesprek-001,Goedemorgen ik heb een vraag over mijn afspraak,Positief,0.734319
1,test-gesprek-001,Ik zie hem nu staan dank je wel!,Positief,0.600299
2,test-gesprek-002,Waarom werkt de bevestigingslink niet?,Neutraal (taakgericht),0.884543
3,test-gesprek-002,Het lukt nog steeds niet en dit is erg frustre...,Negatief (gefrustreerd),0.877925
4,test-gesprek-003,HELP mijn gegevens zijn verdwenen en niemand r...,Boos (paniek),0.878326
5,test-gesprek-004,Wat zijn jullie openingstijden?,Neutraal (taakgericht),0.920491
6,test-gesprek-005,De uitleg was duidelijk en ik kan verder bedankt!,Positief,0.902432


## Stap 8 — Eén sentiment per gesprek kiezen

Voor dit prototype nemen we het **laatste bruikbare bezoekersbericht**. Dat is een eenvoudige, uitlegbare keuze en geeft aan hoe de bezoeker het gesprek verlaat. Een eerder boos bericht kan hierdoor worden vervangen door een later positief bericht; controleer later met Momants of dit de gewenste bedrijfsregel is.

In [8]:
aantallen = (
    bezoekersberichten.groupby("conversation_id")
    .size()
    .rename("aantal_bezoekersberichten")
)

laatste_berichten = (
    bezoekersberichten.sort_values(["conversation_id", "created_at"])
    .groupby("conversation_id", as_index=False)
    .tail(1)
    .set_index("conversation_id")
)

resultaat = aantallen.to_frame().join(
    laatste_berichten[["sentiment_bericht", "zekerheid_bericht"]]
)

resultaat = resultaat.rename(
    columns={
        "sentiment_bericht": "sentiment_gesprek",
        "zekerheid_bericht": "zekerheid",
    }
).reset_index()

resultaat["uitleg"] = resultaat.apply(
    lambda rij: (
        f"Het laatste van {rij['aantal_bezoekersberichten']} bruikbare "
        f"bezoekersberichten is beoordeeld als {rij['sentiment_gesprek']} "
        f"met {rij['zekerheid']:.0%} zekerheid."
    ),
    axis=1,
)

resultaat["zekerheid"] = resultaat["zekerheid"].round(3)

## Stap 9 — De eindtabel bekijken

Dit is de gevraagde lokale output: één rij per gesprek, zonder berichttekst of uitgesloten privacykolommen.

In [9]:
EINDKOLOMMEN = [
    "conversation_id",
    "aantal_bezoekersberichten",
    "sentiment_gesprek",
    "zekerheid",
    "uitleg",
]

resultaat[EINDKOLOMMEN].head(10)

,conversation_id,aantal_bezoekersberichten,sentiment_gesprek,zekerheid,uitleg
0,test-gesprek-001,2,Positief,0.600,Het laatste van 2 bruikbare bezoekersberichten...
1,test-gesprek-002,2,Negatief (gefrustreerd),0.878,Het laatste van 2 bruikbare bezoekersberichten...
2,test-gesprek-003,1,Boos (paniek),0.878,Het laatste van 1 bruikbare bezoekersberichten...
3,test-gesprek-004,1,Neutraal (taakgericht),0.920,Het laatste van 1 bruikbare bezoekersberichten...
4,test-gesprek-005,1,Positief,0.902,Het laatste van 1 bruikbare bezoekersberichten...


## Stap 10 — De labelmapping handmatig controleren

De mapping is een aanname. Met deze losse testzinnen kun je snel bekijken of het model en de vier Momants-labels logisch reageren. Pas gerust alleen de voorbeeldzinnen aan.

In [10]:
controlezinnen = pd.Series(
    [
        "Dank je wel, dit heeft me echt geholpen!",
        "Ik wil graag weten wanneer mijn afspraak is.",
        "Dit werkt alweer niet en ik word er moe van.",
        "HELP, IK BEN ALLES KWIJT EN NIEMAND REAGEERT!",
    ],
    name="testzin",
)

controle_scores = controlezinnen.apply(classificeer_bericht)
pd.concat([controlezinnen, controle_scores], axis=1)

,testzin,modellabel,sentiment_bericht,zekerheid_bericht
0,"Dank je wel, dit heeft me echt geholpen!",Positive,Positief,0.578357
1,Ik wil graag weten wanneer mijn afspraak is.,Neutral,Neutraal (taakgericht),0.909060
2,Dit werkt alweer niet en ik word er moe van.,Negative,Negatief (gefrustreerd),0.802243
3,"HELP, IK BEN ALLES KWIJT EN NIEMAND REAGEERT!",Very Positive,Positief,0.614798
